In [23]:
# Scientific computing.
import pandas as pd
import numpy as np
from collections import Counter

# Machine learning.
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    confusion_matrix,
)
from imblearn.under_sampling import RandomUnderSampler

# File handling.
from pathlib import Path
import json

In [2]:
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
# From Organization.ipynb.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
# From Organization.ipynb
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

In [3]:
# Paste in Likert scales, for sanity.
# There are two different likerts that are used, where the interpretations are slightly different. 
agreement_scale = {
    'Strongly agree': 5,
    'Moderately agree': 4,
    'Neither disagree nor agree': 3,
    'Moderately disagree': 2,
    'Strongly disagree': 1,
    'Unable to judge': None
}
# Agreement with statements is not the same as assessing helpfulness directly. 
# "Moderately helpful" is not the same as "neither disagree nor agree".
helpfulness_scale = {
    'Extremely helpful': 5,
    'Very helpful': 4,
    'Moderately helpful': 3,
    'Slightly helpful': 2,
    'Not at all helpful': 1,
    'Unable to judge': None
}
# Combine both mappings in a dictionary.
rating_encoding = {**agreement_scale, **helpfulness_scale}

## Step one: Primary Pairs.
T_ Columns are paired with matching R_ columns, based on theme. 
 For instance: The open response column with understanding collaboration is paired with the equivalent rating column that also measures collaborative activities. 

In [4]:
# Leader rating columns for averaging.
Leader_R_Cols = [
    "R_Leader_Helps_Understanding_encoded",
    "R_Leader_Subject_Competence_encoded",
    "R_Leader_Has_Plan_encoded",
    "R_Leader_Willing_To_Help_encoded",
]

# T_ column to its matched R_ column.
Primary_Pairs = [
    ("T_Collaboration_Understanding",      "R_Collaborative_Activities_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Helps_Understanding_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Subject_Competence_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Has_Plan_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Willing_To_Help_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Average_encoded"),
    ("T_Other_Suggestions",                "R_Overall_Session_Helpfulness_encoded"),
    ("T_Program_Overall_Suggestions",      "R_Overall_Session_Helpfulness_encoded"),
]

In [5]:
# Create an average leader score metric.and
# T_Leader_Performance_Suggestions corresponds with columns that rate leader performance.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [6]:
# Build Agreement-scale row index per R_ column from Likert_Guide_df.
# Agreement Likerts are the vast majority of responses. Helpfulness Likerts aren't.

def get_agreement_index(r_col):
    # Strip _encoded suffix to match Likert_Guide_df column names.
    col_name = r_col.replace("_encoded", "")
    if col_name == "R_Leader_Average":
        # For averaged column, intersect Agreement rows across all four leader columns.
        indices = None
        for leader_col in Leader_R_Cols:
            agreement_rows = Likert_Guide_df[
                (Likert_Guide_df["Column"] == leader_col.replace("_encoded", "")) &
                (Likert_Guide_df["Scale"] == "Agreement")
            ][["Discipline", "Course_Code", "Semester", "Year"]]
            mask = Survey_df[["Discipline", "Course_Code", "Semester", "Year"]].merge(
                agreement_rows, how="inner"
            ).index
            indices = mask if indices is None else indices.intersection(mask)
        return indices
    # For single columns, filter directly.
    agreement_rows = Likert_Guide_df[
        (Likert_Guide_df["Column"] == col_name) &
        (Likert_Guide_df["Scale"] == "Agreement")
    ][["Discipline", "Course_Code", "Semester", "Year"]]
    mask = Survey_df.merge(
        agreement_rows,
        on=["Discipline", "Course_Code", "Semester", "Year"],
        how="inner"
    ).index
    return mask

In [7]:
# Bag of words builder.
def build_bow(token_lists):
    # Build vocabulary from all token lists.
    vocab = sorted(set(tok for tokens in token_lists for tok in tokens))
    vocab_index = {tok: i for i, tok in enumerate(vocab)}
    # Build count matrix.
    matrix = np.zeros((len(token_lists), len(vocab)), dtype=int)
    for i, tokens in enumerate(token_lists):
        counts = Counter(tokens)
        for tok, count in counts.items():
            if tok in vocab_index:
                matrix[i, vocab_index[tok]] = count
    return matrix, vocab

In [8]:
def get_top_features(nb_model, vocab, n=20):
    # Extract top n tokens most predictive of each class.
    top_features = {}
    for i, class_label in enumerate(nb_model.classes_):
        top_indices = nb_model.feature_log_prob_[i].argsort()[-n:][::-1]
        top_features[class_label] = [vocab[j] for j in top_indices]
    return top_features

In [18]:
def prepare_model_data(t_col, r_col):
    # Get Agreement-scale row indices for this pairing.
    agreement_idx = get_agreement_index(r_col)
    subset = Survey_df.loc[agreement_idx].copy()
    # Combine lemmas and bigrams as features.
    subset["features"] = subset.apply(
        lambda row: list(row[t_col + "_lemmas"]) + list(row[t_col + "_bigrams"]), axis=1
    )
    # Drop rows with no text, no rating, or neutral rating.
    # We throw out threes. Why? They mean "neither agree nor disagree."
    subset = subset.dropna(subset=[r_col])
    subset = subset[subset["features"].apply(len) > 0]
    subset = subset[subset[r_col] != 3]
    # Binarize rating: above 3 is 1, below 3 is 0 (negative sentiment).
    y = (subset[r_col] > 3).astype(int)
    # Check we have enough data and both classes present.
    if len(y) < 50 or y.nunique() < 2:
        return None, None, None
    return subset, y, subset["features"].tolist()

In [11]:
def split_and_train(features, y):
    # Build BOW from combined lemmas and bigrams.
    X, vocab = build_bow(features)

    # 80/20 stratified split.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Compute vocabulary coverage — tokens seen in training that appear in test.
    train_vocab = set(
        tok for tokens in features[:len(y_train)] for tok in tokens
    )
    test_tokens = set(
        tok for tokens in features[len(y_train):] for tok in tokens
    )
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0
    
    # Train model.
    nb = MultinomialNB()
    nb.fit(X_train, y_train)
    return nb, vocab, X_test, y_train, y_test, vocab_coverage

In [13]:
# Function to extract performance metrics.
def evaluate_model(nb, vocab, X_test, y_train, y_test, vocab_coverage, t_col, r_col, label, y):
    # Generate predictions.
    y_pred = nb.predict(X_test)
    y_prob = nb.predict_proba(X_test)[:, 1]
    # Compute confusion matrix values.
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    # Record class balance.
    # Take mean of binaries = the portions of ones.
    pos_pct = round(y.mean() * 100, 1)
    neg_pct = round(100 - pos_pct, 1)

    # Return metrics of request.
    return {
        "Label":          label,
        "T_Col":          t_col,
        "R_Col":          r_col,
        "N_Train":        len(y_train),
        "N_Test":         len(y_test),
        "Pos_Pct":        pos_pct,
        "Neg_Pct":        neg_pct,
        "Vocab_Size":     len(vocab),
        "Vocab_Coverage": vocab_coverage,
        "Accuracy":       round(accuracy_score(y_test, y_pred), 3),
        "F1":             round(f1_score(y_test, y_pred), 3),
        "Precision":      round(precision_score(y_test, y_pred), 3),
        "Recall":         round(recall_score(y_test, y_pred), 3),
        "ROC_AUC":        round(roc_auc_score(y_test, y_prob), 3),
        "MCC":            round(matthews_corrcoef(y_test, y_pred), 3),
        "TP":             int(tp),
        "TN":             int(tn),
        "FP":             int(fp),
        "FN":             int(fn),
        "Top_Features":   get_top_features(nb, vocab),
    }

In [14]:
def run_naive_bayes(t_col, r_col, label):
    # Prepare data.
    # Calls above model preparation function to process data.
    subset, y, features = prepare_model_data(t_col, r_col)
    if subset is None:
        print(f"Skipping {label}.")
        return None
    
    # Split, build BOW, and train.
    nb, vocab, X_test, y_train, y_test, vocab_coverage = split_and_train(features, y)

    # Evaluate and return all metrics.
    return evaluate_model(nb, vocab, X_test, y_train, y_test, vocab_coverage, t_col, r_col, label, y)

In [ ]:
# Store all primary pairings, with each T_ column predicting its matched R_ column.
Primary_Results = []

for t_col, r_col in Primary_Pairs:
    # Build a readable label for printing and the results table.
    # I.e. delete the stuff that has underscores.
    label = f"{t_col.replace('T_', '')} to {r_col.replace('_encoded', '').replace('R_', '')}."
    print(f"Running: {label}")
    result = run_naive_bayes(t_col, r_col, label)
    if result:
        Primary_Results.append(result)
        # Quick inline summary so we can see progress without waiting for the full table.
        print(f"Accuracy: {result['Accuracy']}. ROC_AUC: {result['ROC_AUC']}")
    print()

Running: Collaboration_Understanding → Collaborative_Activities
Accuracy: 0.962. ROC_AUC: 0.761

Running: Leader_Performance_Suggestions → Leader_Helps_Understanding
Accuracy: 0.926. ROC_AUC: 0.735

Running: Leader_Performance_Suggestions → Leader_Subject_Competence
Accuracy: 0.974. ROC_AUC: 0.588

Running: Leader_Performance_Suggestions → Leader_Has_Plan
Accuracy: 0.979. ROC_AUC: 0.536

Running: Leader_Performance_Suggestions → Leader_Willing_To_Help
Accuracy: 0.977. ROC_AUC: 0.62

Running: Leader_Performance_Suggestions → Leader_Average
Accuracy: 0.97. ROC_AUC: 0.607

Running: Other_Suggestions → Overall_Session_Helpfulness
Accuracy: 0.852. ROC_AUC: 0.48

Running: Program_Overall_Suggestions → Overall_Session_Helpfulness
Accuracy: 0.935. ROC_AUC: 0.602



In [ ]:
# Flatten results into a summary DataFrame.
# Top_Features excluded here.
Primary_Summary_df = pd.DataFrame([{
    "Pairing":         r["Label"],
    "N":               r["N_Train"] + r["N_Test"],
    "Vocab Size":      r["Vocab_Size"],
    "Pos %":           r["Pos_Pct"],
    "Neg %":           r["Neg_Pct"],
    "Accuracy":        r["Accuracy"],
    "F1":              r["F1"],
    "Precision":       r["Precision"],
    "Recall":          r["Recall"],
    "ROC AUC":         r["ROC_AUC"],
    "MCC":             r["MCC"],
    "Vocab Coverage":  r["Vocab_Coverage"],
} for r in Primary_Results])

Primary_Summary_df = Primary_Summary_df.set_index("Pairing")
print(Primary_Summary_df.to_string())

                                                                 N  Vocab Size  Pos %  Neg %  Accuracy     F1  Precision  Recall  ROC AUC    MCC  Vocab Coverage
Pairing                                                                                                                                                         
Collaboration_Understanding → Collaborative_Activities       25101       58947   96.7    3.3     0.962  0.981      0.971   0.991    0.761  0.187            66.6
Leader_Performance_Suggestions → Leader_Helps_Understanding  21343       60194   94.4    5.6     0.926  0.961      0.949   0.973    0.735  0.134            62.0
Leader_Performance_Suggestions → Leader_Subject_Competence   22038       61547   97.6    2.4     0.974  0.987      0.977   0.996    0.588  0.078            62.2
Leader_Performance_Suggestions → Leader_Has_Plan             22074       61686   98.2    1.8     0.979  0.989      0.982   0.997    0.536 -0.007            62.6
Leader_Performance_Suggestions → L